In [23]:
import os
import psycopg2
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

url = "https://api.coinpaprika.com/v1/tickers?quotes=USD"

response = requests.get(url)

print(response)

<Response [200]>


In [24]:
data = response.json()

data[0]

{'id': 'btc-bitcoin',
 'name': 'Bitcoin',
 'symbol': 'BTC',
 'rank': 1,
 'total_supply': 20054347,
 'max_supply': 21000000,
 'beta_value': 0.97352,
 'first_data_at': '2010-07-17T00:00:00Z',
 'last_updated': '2026-07-10T05:31:15Z',
 'quotes': {'USD': {'price': 63929.09954271844,
   'volume_24h': 25291327209.24289,
   'volume_24h_change_24h': 9.329999923706055,
   'market_cap': 1282056345627,
   'market_cap_change_24h': 1.909999966621399,
   'percent_change_15m': -0.019999999552965164,
   'percent_change_30m': -0.03999999910593033,
   'percent_change_1h': -0.14000000059604645,
   'percent_change_6h': 1.100000023841858,
   'percent_change_12h': 1.5299999713897705,
   'percent_change_24h': 1.909999966621399,
   'percent_change_7d': 3.6600000858306885,
   'percent_change_30d': 0,
   'percent_change_1y': 0,
   'ath_price': 126173.1777846797,
   'ath_date': '2025-10-06T19:00:40Z',
   'percent_from_price_ath': -49.33}}}

In [25]:


# coin_df = pd.DataFrame(data)

# coin_df.head(5)

rows = []
for coin in data:
    if coin["symbol"] in ["BTC","ETH","USDT","BNB","USDC","GAL","UTK","MAPO","BTX"]:
        rows.append({
            "id": coin["id"],
            "name":coin["name"],
            "symbol": coin["symbol"],
            "rank": coin["rank"],
            "last_update": coin["last_updated"],
            "price_usd" : coin["quotes"]["USD"]["price"],
            "mrkt_cap" : coin["quotes"]["USD"]["market_cap"]

        })

# rows
coins_df = pd.DataFrame(rows)

coins_df

,id,name,symbol,rank,last_update,price_usd,mrkt_cap
0,btc-bitcoin,Bitcoin,BTC,1,2026-07-10T05:31:15Z,63929.099543,1282056345627
1,eth-ethereum,Ethereum,ETH,2,2026-07-10T05:31:15Z,1772.661190,213475056661
2,usdt-tether,Tether,USDT,3,2026-07-10T05:31:15Z,0.999371,184191262488
3,bnb-binance-coin,BNB,BNB,4,2026-07-10T05:31:15Z,576.003435,77635284893
4,usdc-usd-coin,USDC,USDC,5,2026-07-10T05:31:15Z,0.999986,73291108030
5,usdc-ccip-bridged-usdc-ronin,CCIP Bridged USDC (Ronin),USDC,624,2026-07-10T05:31:15Z,0.998630,19185247
6,map-map-protocol,MAP Protocol,MAPO,987,2026-07-10T05:31:15Z,0.001130,5825038
7,gal-galatasaray-fan-token,Galatasaray Fan Token,GAL,989,2026-07-10T05:31:15Z,0.885016,5813266
8,utk-utrust,xMoney,UTK,993,2026-07-10T05:31:15Z,0.008165,5749240
9,btx-beatswap,BeatSwap,BTX,1039,2026-07-10T05:31:15Z,0.022433,5041861


In [26]:
coins_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           14 non-null     str    
 1   name         14 non-null     str    
 2   symbol       14 non-null     str    
 3   rank         14 non-null     int64  
 4   last_update  14 non-null     str    
 5   price_usd    14 non-null     float64
 6   mrkt_cap     14 non-null     int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 916.0 bytes


In [27]:
coins_df["last_update"] = pd.to_datetime(coins_df["last_update"])
coins_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   id           14 non-null     str                
 1   name         14 non-null     str                
 2   symbol       14 non-null     str                
 3   rank         14 non-null     int64              
 4   last_update  14 non-null     datetime64[us, UTC]
 5   price_usd    14 non-null     float64            
 6   mrkt_cap     14 non-null     int64              
dtypes: datetime64[us, UTC](1), float64(1), int64(2), str(3)
memory usage: 916.0 bytes


In [28]:
DATABASE_HOST= os.getenv("DATABASE_HOST")
DATABSE_PORT = os.getenv("DATABASE_PORT")
DATABASE_NAME= os.getenv("DATABASE_NAME")
DATABASE_USER = os.getenv("DATABASE_USER")
DATABASE_PASSWORD = os.getenv("DATABASE_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DATABASE_USER}:{DATABASE_PASSWORD}@{DATABASE_HOST}:{DATABSE_PORT}/{DATABASE_NAME}")


In [29]:
coins_df.to_sql(
    "paprika_coins",
    con=engine,
    if_exists="replace",
    index=False
)

14

In [30]:
with engine.connect() as conn:
    resort = conn.execute(text("select * from paprika_coins;"))
    for i in resort:
        print(i)

('btc-bitcoin', 'Bitcoin', 'BTC', 1, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 63929.09954271844, 1282056345627)
('eth-ethereum', 'Ethereum', 'ETH', 2, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 1772.6611902823176, 213475056661)
('usdt-tether', 'Tether', 'USDT', 3, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 0.9993708427173171, 184191262488)
('bnb-binance-coin', 'BNB', 'BNB', 4, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 576.0034347275717, 77635284893)
('usdc-usd-coin', 'USDC', 'USDC', 5, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 0.9999863436388444, 73291108030)
('usdc-ccip-bridged-usdc-ronin', 'CCIP Bridged USDC (Ronin)', 'USDC', 624, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=datetime.timezone.utc), 0.9986298183100213, 19185247)
('map-map-protocol', 'MAP Protocol', 'MAPO', 987, datetime.datetime(2026, 7, 10, 5, 31, 15, tzinfo=dat